
`Shapespyer` is a Python toolkit that builds starting configurations for molecular dynamics (MD) simulations of soft matter, such as micelles, vesicles and bilayers in solvent, and equilibrates the systems with GROMACS. We're developing a desktop app `EasyShapes` that uses `shapes.project`, a Shapespyer package, as its backend for building and equilibrating such systems. The notebooks for our section today use `shapes.project` directly. It provides pythonic API that could be used in jupyter, python scripts and GUI, but the same functionality (and more) is available with Shapespyer's command-line tools (`smiles`, `shape`, `gmx-*`). While the command-line tools suit batch runs from shell scripts and cluster job queues, `shapes.project` suits interactive work.

#### Section program following the `EasyShapes` workflow and `shapes.project` layout

The notebooks of this workshop section follow the layout of `shapes.project`. In order to equilibrate a system, a project needs a sample model. A sample model, in turn, needs a structure and a buffer, and a structure needs components with their counter-ions. The notebooks build this up in the opposite direction, one step at a time, from left to right in the diagram:

```
┌─ Library ────────────────────────────────────────────────────────────────────────────────┐
│ Assets downloaded from Shapespyer repo or saved by the user                              │
└─────────────┬──▲───────────────────┬──▲────────────────────────┬──▲───────────────────┬──┘
          load│  │save           load│  │save                load│  │save           load│
              ▼  │                   ▼  │                        ▼  │                   ▼
╔═ Component ══════╗   ┌─ Structure ─────────┐   ┌─ SampleModel ──────┐   ┌─ Project ──────┐
║ .paths           ║──►│ .components         │──►│ .structure         │──►│ .sample_model  │
║ .counter_ion     ║   │ .layers (Ball)      │   │ .buffer            │   │ .analysis      │
║ .mint            ║   │ .lamellae (Vesicle) │   │                    │   │                │
║ .mext            ║   │ .motif (Lattice)    │   │                    │   │                │
╚══════════════════╝   └─────────────────────┘   └────────────────────┘   └────────────────┘
  notebook 1             notebook 2                notebook 3               notebook 4
└─────────────────── EasyShapes: Sample Model tab ────────────────────┘   Project + Analysis tabs
```

0. **Library** (all notebooks) is represented by the class `Library`, which manages the asset files, both the ready-made molecules and equilibration settings downloaded on first use and the assets you save. Every class that holds files has `load(name)` and `save()`, which use the `Library` object underneath.
1. → **Component** (notebook 1) is represented by the classes `Lipid` and `Surfactant`. Components are loaded from the library or created from a structure file or a SMILES string.
2. **Structure** (notebook 2) is represented by the classes `Ring`, `Rod`, `Bilayer`, `Monolayer`, `Ball`, `Vesicle` and `Lattice`, each arranging copies of components in its shape.
3. **SampleModel** (notebook 3) is a structure in a buffer of solvent and salt, solvated with GROMACS. The buffer is represented by the classes `Buffer`, `Solvent` and `Salt`.
4. **Project** (notebook 4) holds a sample model and its analysis, the staged GROMACS equilibration that relaxes the model.

# 1. Components

Components are the molecules that a structure is built from, represented by `shapes.project` classes `Lipid` and `Surfactant`. They work with the library by name. `Surfactant.load("SDS")` takes SDS from the library, downloading it the first time. A component you make yourself, from your own file or a SMILES string, goes into the library with `save()`, and from then on every notebook can load it by name, as it loads SDS. Everything else that holds files works with the library the same way, with `load(name)` and `save()`. That includes ions and solvents, represented by `Ion` and `Solvent`, which go into the buffer or serve as counter-ions, and also structures, buffers and sample models.

This notebook works with the components and the library.

```
┌─ Library ────────┐
└─────────────┬──▲─┘
          load│  │save
              ▼  │
╔═ Component ══════╗   ┌─ Structure ──┐
║ .paths           ║──►│ .components  │──► …
║ .counter_ion     ║   └──────────────┘
║ .mint            ║
║ .mext            ║
╚══════════════════╝
```

In this notebook you will:

1. see the assets in the Shapespyer library
2. load a component from the library and look at it
3. make a component from a file of your own
4. make a component from a SMILES string
5. save your components to the library, so every notebook can load them

We have to start with a few setup cells. Make sure Shapespyer is installed in your virtual environment. [Shapespyer installation docs](https://simnavi.gitlab.io/shapespyer.io/installation.html)

In [ ]:
# Setup: check that shapespyer.yaml is next to the notebook
from pathlib import Path

CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files
if not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

The next cell provides helper functions that only download an example file and check the library.
These helper functions, as well as the helpers in other notebooks, **are not part of the Shapespyer workflow**. They are all in the `Helpers` class, and the notebook always calls them from it, as `Helpers.<name>()`, so you can tell them apart from the Shapespyer package.

In [ ]:
# Helpers (not part of the Shapespyer workflow)
import urllib.request


class Helpers:
    """Workshop helpers, not part of Shapespyer: call them as Helpers.<name>()."""

    EXAMPLE_FILES_URL = "https://gitlab.com/simnavi/shapespyer/-/raw/v0.3.1/assets/components"

    @staticmethod
    def download_example_file(path, folder):
        """Download a file of the Shapespyer repository into a folder, to stand in for a file of your own."""
        folder.mkdir(parents=True, exist_ok=True)
        target = folder / path.rsplit("/", 1)[-1]
        urllib.request.urlretrieve(f"{Helpers.EXAMPLE_FILES_URL}/{path}", target)
        return target

    @staticmethod
    def show_library_entry(library, asset_type, name):
        """Print whether you saved a library entry yourself, and its files."""
        record = library.find(asset_type, name=name)
        print(f"{name} saved by you:", "ser_data" in record)
        for path in record.get("paths", []):
            print(" ", path)

In the setup we pass the `shapespyer.yaml` config file, which says where Shapespyer keeps its files. Here it keeps the workshop's files apart from your own Shapespyer data. In the common scenario, users won't have to use the config file, as the `shapes.project` package manages files automatically. Use the config if you want manual control over where the files are kept.

In [ ]:
# Config file, log level and work folder
from shapes.basics.utils import LogConfiguration
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Workshop files:", config.app_path)

# the level applies to Shapespyer's own handlers; stop the messages there, so no other handler prints them again
LogConfiguration("WARNING").logger.propagate = False

# the downloaded example file goes to a work folder, which the clean up deletes
WORK = config.app_path / "work"

In [ ]:
from shapes.project import Ion, Library, Surfactant

### The library

Here you choose where the library keeps its files: both the assets downloaded from the Shapespyer assets and the components you save yourself. We give `LocalAssets/` in the application directory, which is also the default. Here that is `.data/LocalAssets` in this folder.

The first `Library()` of a session reads the remote index of assets and downloads their files into that folder. Only that first call sets the path: later calls return the same library.

What you will use from the library:

- `available_assets`: the names of every asset available, local or remote, by asset type.
- `has_entry(asset_type, name)`: tells whether a name is taken in the library.
- `delete_entry(asset_type, name, force=False)`: deletes what the library stores for an entry. `force=True` deletes it even though assets of this session hold its files.
- `sync(all_remote=False, force=False)`: updates the local copies of remote assets to the remote index. `all_remote=True` also downloads the remote assets you have no copy of yet; `force=True` also replaces the assets you saved yourself with the remote ones.
- `clean_temp(force=False)`: deletes the dir, which holds the working files of assets and starts it afresh.
- `available_projects`, `available_examples` and `delete_project(path, force=False)`: the saved projects, the example projects, and deleting a project. Notebook 4 uses them.

Loading and saving go through the components themselves: `Surfactant.load(name)` and `.save()`, below.

These are the assets ready to use, by type:

In [ ]:
library = Library(path=config.app_path / "LocalAssets")
library.available_assets

### Load a component and look at it

`Surfactant.load(name)` takes a surfactant from the library; `Lipid`, `Ion` and `Solvent` load the same way. A surfactant, like a lipid, holds:

- `name`: the name of the component, also the residue name of its molecules in the structure file.
- `paths`: the files of the component: its structure file (`.gro` or `.pdb`) and its topology (`.itp`).
- `counter_ion`: the counter ion, or the name of one to load from the library. We provide one further.
- `mint` and `mext`: the indices of the interior and the exterior (back-)bone atoms, closer to and farther from the structure centre. A structure uses them to place the molecule; notebook 2 explains them in detail.

The component's `.show()` draws the molecule. It must be the last line of the cell.

In [ ]:
sds = Surfactant.load("SDS")
sds.show()

### Give a component its counter ion

`counter_ion` says which ion goes with the component. The ion is loaded from the library, like the component: `available_assets["Ion"]` above lists the ions, and `NAT` is sodium. When a sample model is assembled, the counter ions go into the box with the component; notebook 4 does that.

In [ ]:
sds.counter_ion = Ion.load("NAT")
sds.counter_ion.paths

### A component from your own file

Here `Helpers.download_example_file()` downloads a `.gro` file from the Shapespyer repository, as if it were your own. The name of the component must match the residue name in the file. This file also holds an SDS molecule next to the CTA, and `.show()` draws only the residues named CTA.

In [ ]:
my_file = Helpers.download_example_file("surfactants/CTA/CTA.gro", WORK)

cta = Surfactant(name="CTA", paths=[my_file])
cta.show()

### A component from a SMILES string

Here we make SDS from its SMILES string. Hydrogens are added, and the atoms follow the SMILES order. We write the tail first, so the "bone" of the molecule runs from the tail end (`mint` is 0) to the head (`mext` is the last atom), as a micelle needs.

The name becomes the residue name in the file, at most 5 characters. We call it `SDSsm`, so it doesn't clash with the library's SDS. A structure finds its molecules by this residue name, so a component can't be renamed later. The component gets a structure file only, with no topology. So it can be built into structures, but not equilibrated.

`from_smiles` takes:

- `name`: the name of the component, also the residue name in the file, at most 5 characters.
- `smiles`: the SMILES string of one molecule, of two atoms or more.
- `box`: the box lengths along X, Y and Z of the written file, in nm. 3 nm on each side is more than one SDS molecule is long.
- `formula`: the chemical formula for the title line of the file. Without it, the name is used.
- `cis`: kinks C=C double bonds as cis instead of trans. SDS has none.
- `align_z`: rotates the molecule so a terminal bond lies along Z.
- `flatten_xz`: flattens the aligned molecule into the XZ plane; it works only with `align_z`.
- `extension`: the format of the structure file, `.gro` or `.pdb`.

In [ ]:
sds_from_smiles = Surfactant.from_smiles(
    name="SDSsm",
    smiles="CCCCCCCCCCCCOS(=O)(=O)[O-]",
    box=(3.0, 3.0, 3.0),  # in nm
    formula="C12H25SO4",
    cis=False,
    align_z=True,
    flatten_xz=False,
    extension=".gro",
)
print(sds_from_smiles.number_of_atoms, sds_from_smiles.mint, sds_from_smiles.mext)
sds_from_smiles.show()

### Save your components to the library

`.save()` copies the files of a component into the library, `LocalAssets/<type>/<name>/`, with a `ser_data.yaml` record of the whole component. From then on, `Surfactant.load(name)` finds it: in this notebook, in the other notebooks and in later sessions.

We save our two components:

- `SDSsm` has a new name, so it is simply added.
- `CTA` has the name of a library asset, so saving it replaces that asset in your local library.

If you run this notebook a second time without the clean up at the end, use `overwrite=True` here too.

In [ ]:
sds_from_smiles.save()
library.available_assets["Surfactant"]

The name `CTA` is taken by the library's CTA, so a plain `.save()` refuses:

In [ ]:
Helpers.show_library_entry(library, "Surfactant", "CTA")
try:
    cta.save()
except ValueError as err:
    print(err)

`overwrite=True` replaces it. The library looks at your local entries first, and `sync()` leaves the entries you saved alone, so every notebook now loads your CTA. Here your file is the same as the library's; with a CTA of your own, such as one with other coordinates, yours would be used.

In [ ]:
cta.save(overwrite=True)
Helpers.show_library_entry(library, "Surfactant", "CTA")

Load a saved component back from the library, as any other notebook would. It works on the copies in the library, not on the files it was made from:

In [ ]:
loaded = Surfactant.load("SDSsm")
print(loaded.paths)
loaded.show()

### Clean up (optional)

This deletes the work folder with the downloaded `CTA.gro`, and your two library entries. Without this, every later notebook and session would load your CTA instead of the library's. `force=True` is needed, because `loaded` still works on the files of its entry. The library's CTA comes back with the next lookup, which downloads it again.

In [ ]:
import shutil

shutil.rmtree(WORK, ignore_errors=True)
library.delete_entry("Surfactant", "SDSsm", force=True)
library.delete_entry("Surfactant", "CTA", force=True)

Helpers.show_library_entry(library, "Surfactant", "CTA")
print(WORK.exists(), library.available_assets["Surfactant"])